# 버스 정류장 이벤트 데이터 전처리

이 노트북은 버스 정류장 이벤트 데이터를 불러와 분석에 용이한 형태로 가공하는 과정을 담고 있습니다.

**주요 작업:**
1.  **데이터 로드**: 원본 데이터를 불러와 구조를 확인합니다.
2.  **시간 데이터 분리**: `YYYYMMDDHHMMSS` 형식의 단일 시간 필드를 연, 월, 일, 시 등 세부 필드로 분리합니다.
3.  **데이터 병합 및 저장**: 분리된 시간 데이터를 원본 데이터와 병합하여 새로운 CSV 파일로 저장합니다.
4.  **결과 확인**: 최종 결과물을 출력하여 작업 완료를 검증합니다.

## 1. 라이브러리 설치 및 데이터 로드

In [1]:
# 데이터 분석에 필요한 라이브러리를 설치합니다.
!pip3 install pandas tqdm

import pandas as pd
from datetime import datetime
from tqdm import tqdm

# 원본 버스 이벤트 데이터를 불러옵니다.
masterDF = pd.read_csv("./data/_gs_busevent.csv")

# 데이터의 기본 정보와 상위 5개 행을 확인합니다.
masterDF.info()
masterDF.head()


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3 -m pip install --upgrade pip
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202935 entries, 0 to 202934
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   route_name         202935 non-null  object 
 1   bstop_arrive_time  202935 non-null  int64  
 2   bstop_leave_time   202935 non-null  int64  
 3   stop_time          202935 non-null  int64  
 4   node_name          202935 non-null  object 
 5   node_x_pos         202935 non-null  float64
 6   node_y_pos         202935 non-null  float64
dtypes: float64(2), int64(3), object(2)
memory usage: 10.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202935 entries, 0 to 202934
Data columns (total 7 columns):
 #   Column             Non-Null

,route_name,bstop_arrive_time,bstop_leave_time,stop_time,node_name,node_x_pos,node_y_pos
0,818,20200104000053,20200104000140,49,하양초등학교,128.81746,35.91229
1,818,20200104000214,20200104000240,27,하양대구은행,128.82080,35.91398
2,818,20200104000501,20200104000525,25,금락초교 건너,128.82316,35.91077
3,818,20200104000554,20200104000620,27,부기리LH아파트,128.82388,35.90677
4,818,20200104000648,20200104000658,11,부기2리,128.82430,35.90354


## 2. 시간 데이터 분리

`bstop_arrive_time`과 `bstop_leave_time` 컬럼을 분석에 용이하도록 연, 월, 일, 시, 분, 초 등의 개별 컬럼으로 분리합니다.

In [2]:
# 시간 정보를 분리하여 저장할 리스트를 초기화합니다.
bstop_arrive_list = []
bstop_leave_list = []

# 도착 시간을 분리합니다.
for time in tqdm(masterDF['bstop_arrive_time'], desc="도착 시간 처리 중"):
    time_data = datetime.strptime(str(time), "%Y%m%d%H%M%S")
    bstop_arrive_list.append({
        "bstop_arrive_year": time_data.year,
        "bstop_arrive_month": time_data.month,
        "bstop_arrive_day": time_data.day,
        "bstop_arrive_days": time_data.strftime("%Y-%m-%d"),
        "bstop_arrive_hour": time_data.hour,
        "bstop_arrive_minute": time_data.minute,
        "bstop_arrive_second": time_data.second,
        "bstop_arrive_timezone": "UTC+9",
    })

# 출발 시간을 분리합니다.
for time in tqdm(masterDF['bstop_leave_time'], desc="출발 시간 처리 중"):
    time_data = datetime.strptime(str(time), "%Y%m%d%H%M%S")
    bstop_leave_list.append({
        "bstop_leave_year": time_data.year,
        "bstop_leave_month": time_data.month,
        "bstop_leave_day": time_data.day,
        "bstop_leave_days": time_data.strftime("%Y-%m-%d"),
        "bstop_leave_hour": time_data.hour,
        "bstop_leave_minute": time_data.minute,
        "bstop_leave_second": time_data.second,
        "bstop_leave_timezone": "UTC+9",
    })

# 분리된 시간 데이터를 데이터프레임으로 변환합니다.
df_arrive = pd.DataFrame(bstop_arrive_list)
df_leave = pd.DataFrame(bstop_leave_list)

# 도착 시간과 출발 시간 데이터프레임을 하나로 합칩니다.
df_time = pd.concat([df_arrive, df_leave], axis=1)
df_time.head()

출발 시간 처리 중: 100%|██████████| 202935/202935 [00:02<00:00, 90164.66it/s]



,bstop_arrive_year,bstop_arrive_month,bstop_arrive_day,bstop_arrive_days,bstop_arrive_hour,bstop_arrive_minute,bstop_arrive_second,bstop_arrive_timezone,bstop_leave_year,bstop_leave_month,bstop_leave_day,bstop_leave_days,bstop_leave_hour,bstop_leave_minute,bstop_leave_second,bstop_leave_timezone
0,2020,1,4,2020-01-04,0,0,53,UTC+9,2020,1,4,2020-01-04,0,1,40,UTC+9
1,2020,1,4,2020-01-04,0,2,14,UTC+9,2020,1,4,2020-01-04,0,2,40,UTC+9
2,2020,1,4,2020-01-04,0,5,1,UTC+9,2020,1,4,2020-01-04,0,5,25,UTC+9
3,2020,1,4,2020-01-04,0,5,54,UTC+9,2020,1,4,2020-01-04,0,6,20,UTC+9
4,2020,1,4,2020-01-04,0,6,48,UTC+9,2020,1,4,2020-01-04,0,6,58,UTC+9


## 3. 데이터 병합 및 저장

In [3]:
# 원본 데이터프레임과 분리된 시간 정보 데이터프레임을 병합합니다.
masterDF_preprocessed = pd.concat([masterDF, df_time], axis=1)

# 전처리된 데이터를 CSV 파일로 저장합니다.
masterDF_preprocessed.to_csv("./data/_gs_busevent_preprocessed.csv", index=False)

print("데이터 전처리 및 저장이 완료되었습니다.")
masterDF_preprocessed.head()

데이터 전처리 및 저장이 완료되었습니다.


,route_name,bstop_arrive_time,bstop_leave_time,stop_time,node_name,node_x_pos,node_y_pos,bstop_arrive_year,bstop_arrive_month,bstop_arrive_day,...,bstop_arrive_second,bstop_arrive_timezone,bstop_leave_year,bstop_leave_month,bstop_leave_day,bstop_leave_days,bstop_leave_hour,bstop_leave_minute,bstop_leave_second,bstop_leave_timezone
0,818,20200104000053,20200104000140,49,하양초등학교,128.81746,35.91229,2020,1,4,...,53,UTC+9,2020,1,4,2020-01-04,0,1,40,UTC+9
1,818,20200104000214,20200104000240,27,하양대구은행,128.82080,35.91398,2020,1,4,...,14,UTC+9,2020,1,4,2020-01-04,0,2,40,UTC+9
2,818,20200104000501,20200104000525,25,금락초교 건너,128.82316,35.91077,2020,1,4,...,1,UTC+9,2020,1,4,2020-01-04,0,5,25,UTC+9
3,818,20200104000554,20200104000620,27,부기리LH아파트,128.82388,35.90677,2020,1,4,...,54,UTC+9,2020,1,4,2020-01-04,0,6,20,UTC+9
4,818,20200104000648,20200104000658,11,부기2리,128.82430,35.90354,2020,1,4,...,48,UTC+9,2020,1,4,2020-01-04,0,6,58,UTC+9


## 4. 결과 확인

In [4]:
# 저장된 파일을 다시 불러와 상위 5개 행을 사전 형태로 출력하여 최종 결과를 검증합니다.
pd.read_csv("./data/_gs_busevent_preprocessed.csv").head().to_dict()

{'route_name': {0: '818', 1: '818', 2: '818', 3: '818', 4: '818'},
 'bstop_arrive_time': {0: 20200104000053,
  1: 20200104000214,
  2: 20200104000501,
  3: 20200104000554,
  4: 20200104000648},
 'bstop_leave_time': {0: 20200104000140,
  1: 20200104000240,
  2: 20200104000525,
  3: 20200104000620,
  4: 20200104000658},
 'stop_time': {0: 49, 1: 27, 2: 25, 3: 27, 4: 11},
 'node_name': {0: '하양초등학교',
  1: '하양대구은행',
  2: '금락초교 건너',
  3: '부기리LH아파트',
  4: '부기2리'},
 'node_x_pos': {0: 128.81746,
  1: 128.8208,
  2: 128.82316,
  3: 128.82388,
  4: 128.8243},
 'node_y_pos': {0: 35.91229,
  1: 35.91398,
  2: 35.91077,
  3: 35.90677,
  4: 35.90354},
 'bstop_arrive_year': {0: 2020, 1: 2020, 2: 2020, 3: 2020, 4: 2020},
 'bstop_arrive_month': {0: 1, 1: 1, 2: 1, 3: 1, 4: 1},
 'bstop_arrive_day': {0: 4, 1: 4, 2: 4, 3: 4, 4: 4},
 'bstop_arrive_days': {0: '2020-01-04',
  1: '2020-01-04',
  2: '2020-01-04',
  3: '2020-01-04',
  4: '2020-01-04'},
 'bstop_arrive_hour': {0: 0, 1: 0, 2: 0, 3: 0, 4: 0},
 'bstop_